LABORATORIO 3 - Vocabulario

Toma documentos_preprocesados.txt (salida del laboratorio 2, formato: Numero | titulo | texto),
obtiene el vocabulario de la coleccion CACM, lo reduce y escribe ambos vocabularios
ordenados alfabeticamente en vocabulario.txt y vocabularioReducido.txt.

Mecanismo de reduccion: filtro por frecuencia de documento (df = numero de documentos
en los que aparece un termino).
- Se quitan los terminos con df = 1: aparecen en un solo documento, suelen ser nombres propios,
  siglas o errores de escritura y casi no ayudan a relacionar documentos con consultas.
- Se quitan los terminos con df > 20% de los documentos: aparecen en muchisimos documentos, por lo que
  no sirven para distinguir un documento de otro (funcionan como stopwords del dominio,
  por ejemplo algorithm, comput, program, system).

In [1]:
import os

In [2]:
from collections import Counter

In [3]:
# carpeta de la practica 2 (entrada) y de la practica 3 (salida)
carpeta_entrada = r"C:\Users\erick\OneDrive\Documentos\documentos UNI\Semestre 8\Recuperacion de info\Practicas en clase\Practica2"
corpus_root = r"C:\Users\erick\OneDrive\Documentos\documentos UNI\Semestre 8\Recuperacion de info\Practicas en clase\Practica3"

LECTURA DE LOS DOCUMENTOS PREPROCESADOS

In [4]:
with open(os.path.join(carpeta_entrada, "documentos_preprocesados.txt"), encoding="utf-8") as f:
    lineas = f.read().splitlines()

In [18]:
print(lineas[19])  # ejemplo de documento

20 | acceler converg iter process | techniqu discuss appli iter procedur solut equat acceler rate converg iter converg induc converg iter diverg illustr exampl given


In [5]:
# cada linea es "Numero | titulo | texto"; se juntan titulo y texto como las palabras del documento
documentos = {}
for linea in lineas:
    num, titulo, texto = linea.split(" | ", 2)
    documentos[num] = (titulo + " " + texto).split()

In [17]:
print(documentos["20"])  # ejemplo de documento

['acceler', 'converg', 'iter', 'process', 'techniqu', 'discuss', 'appli', 'iter', 'procedur', 'solut', 'equat', 'acceler', 'rate', 'converg', 'iter', 'converg', 'induc', 'converg', 'iter', 'diverg', 'illustr', 'exampl', 'given']


In [6]:
N = len(documentos)
print(N, "documentos leidos")
print(documentos["20"])

3204 documentos leidos
['acceler', 'converg', 'iter', 'process', 'techniqu', 'discuss', 'appli', 'iter', 'procedur', 'solut', 'equat', 'acceler', 'rate', 'converg', 'iter', 'converg', 'induc', 'converg', 'iter', 'diverg', 'illustr', 'exampl', 'given']


VOCABULARIO COMPLETO

El vocabulario es el conjunto de terminos distintos de toda la coleccion.

In [7]:
# frecuencia total de cada termino (tf) y numero de documentos donde aparece (df)
tf = Counter()
df = Counter()
for palabras in documentos.values():
    tf.update(palabras)
    df.update(set(palabras))   # set: cada documento cuenta una sola vez

In [8]:
vocabulario = sorted(tf)   # ordenado alfabeticamente
print("Longitud del vocabulario:", len(vocabulario))

Longitud del vocabulario: 5279


In [9]:
print("Primeros 50 terminos:", vocabulario[:50])
print("Terminos mas frecuentes (df):", df.most_common(20))

Primeros 50 terminos: ['a', 'aa', 'aaron', 'ab', 'abac', 'abacus', 'abandon', 'abbott', 'abbrevi', 'abcd', 'abd', 'abil', 'abl', 'aboot', 'abraham', 'abscissa', 'absenc', 'absolut', 'absorb', 'absorpt', 'abstract', 'academ', 'academician', 'acceler', 'accent', 'accept', 'access', 'accid', 'accident', 'accommod', 'accompani', 'accomplish', 'accord', 'account', 'accru', 'accumul', 'accur', 'accuraci', 'acf', 'achiev', 'acknowledg', 'acl', 'acm', 'acosbxc', 'acqui', 'acquir', 'acquisit', 'acronym', 'across', 'act']
Terminos mas frecuentes (df): [('algorithm', 1308), ('comput', 863), ('use', 840), ('program', 760), ('system', 675), ('present', 518), ('describ', 506), ('method', 502), ('problem', 449), ('paper', 442), ('time', 412), ('data', 367), ('languag', 364), ('process', 364), ('given', 360), ('discuss', 356), ('number', 353), ('techniqu', 339), ('function', 334), ('oper', 331)]


REDUCCION DEL VOCABULARIO (filtro por frecuencia de documento)

In [10]:
df_min = 2          # minimo de documentos en los que debe aparecer un termino
df_max = 0.2 * N    # maximo: el 20% de la coleccion (con 50% no se eliminaba ningun termino)

In [11]:
# terminos que se eliminan por cada regla, para documentar el efecto de cada una
raros = [w for w in vocabulario if df[w] < df_min]
comunes = [w for w in vocabulario if df[w] > df_max]
print("Terminos con df <", df_min, ":", len(raros))
print("Ejemplos:", raros[:30])
print("Terminos con df >", df_max, ":", len(comunes), comunes)

Terminos con df < 2 : 2258
Ejemplos: ['aa', 'aaron', 'abac', 'abacus', 'abbott', 'abcd', 'abd', 'aboot', 'abraham', 'absorb', 'absorpt', 'academician', 'accent', 'accid', 'accident', 'acf', 'acknowledg', 'acl', 'acosbxc', 'acqui', 'acronym', 'activitybackward', 'actor', 'acycl', 'ada', 'additionsand', 'adept', 'adher', 'adjoin', 'adjoint']
Terminos con df > 640.8000000000001 : 5 ['algorithm', 'comput', 'program', 'system', 'use']


In [ ]:
vocabulario_reducido = [w for w in vocabulario if df_min <= df[w] <= df_max]
print("Longitud del vocabulario reducido:", len(vocabulario_reducido))
print("Reduccion: %.2f%%" % (100 * (1 - len(vocabulario_reducido) / len(vocabulario))))

Longitud del vocabulario reducido: 3016
Reduccion: 42.87%


ESCRITURA DE LOS VOCABULARIOS (un termino por linea, orden alfabetico)

In [13]:
def escribir_vocabulario(terminos, nombre_salida):
    with open(os.path.join(corpus_root, nombre_salida), "w", encoding="utf-8") as f:
        f.write("\n".join(sorted(terminos)))
    print(len(terminos), "terminos ->", nombre_salida)

In [14]:
escribir_vocabulario(vocabulario, "vocabulario.txt")
escribir_vocabulario(vocabulario_reducido, "vocabularioReducido.txt")

5279 terminos -> vocabulario.txt
3016 terminos -> vocabularioReducido.txt
